# CityBrain v3 — Interaction Features + 1D-CNN + Gated Fusion
**COMP 9130 Group 5 | Vancouver Pavement Risk Assessment**

### Changes from v2:
| # | Change | Why |
|---|--------|-----|
| 1 | **Weather × Slope/Streetuse interaction channels** in temporal tensor | Break weather's "global uniformity" — same rain, different damage on steep vs flat roads |
| 2 | **1D-CNN replaces BiLSTM** (32d embedding, not 128d) | Short 12-step sequence suits CNN better; 75% fewer params; less noise injected into fusion |
| 3 | **Gated Attention Fusion** replaces naive concat | Model learns per-sample branch weights — auto-downweights useless branches |
| 4 | **Removed auxiliary PCI regression head** | PCI score = label in different form, not a true auxiliary task |
| 5 | **Removed `neigh_high_risk_pct`** | Target leakage |

---

In [ ]:
# ============================================================
# 0. Setup
# ============================================================
import os, ast, json, warnings, pickle
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from shapely.geometry import shape, Point
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/AI-FinalProject/data'

# DATA_DIR = './data'  # local
SNAP_RADIUS_M = 150
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

LABEL_MAP = {'VERY GOOD': 0, 'GOOD': 0, 'FAIR': 1, 'POOR': 2, 'VERY POOR': 2}

def safe_load(filename, **kwargs):
    path = os.path.join(DATA_DIR, filename) if not os.path.isabs(filename) else filename
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        first = f.readline()
    sep = ';' if first.count(';') > first.count(',') else ','
    return pd.read_csv(path, sep=sep, on_bad_lines='skip', **kwargs)

Mounted at /content/drive
Device: cpu


## 1. Load & Merge Pavement Data

In [ ]:
enriched_path = os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/pavement_enriched.csv')
if os.path.exists(enriched_path):
    df = pd.read_csv(enriched_path)
    print(f'Loaded pavement_enriched.csv: {len(df):,} rows')
else:
    df_local = safe_load('/content/drive/MyDrive/AI-FinalProject/data/pavement_condition.csv'); df_local['road_type'] = 'local'
    df_major = safe_load('/content/drive/MyDrive/AI-FinalProject/data/pavement_condition_major_road_network.csv'); df_major['road_type'] = 'major'
    df = pd.concat([df_local, df_major], ignore_index=True)
    coords = df['geo_point_2d'].str.split(',', expand=True).astype(float)
    df['lat'], df['lon'] = coords[0], coords[1]

df = df[df['PCI Rating'].isin(LABEL_MAP)].copy()
df['risk_label'] = df['PCI Rating'].map(LABEL_MAP)
df['source'] = (df['road_type'] == 'major').astype(int)
print(f'After label mapping: {len(df):,}')
print(df['risk_label'].value_counts().sort_index())

Loaded pavement_enriched.csv: 13,764 rows
After label mapping: 13,764
risk_label
0    6080
1    3061
2    4623
Name: count, dtype: int64


## 2. Static Features

In [ ]:
pave_coords = np.column_stack([df['lat'].values, df['lon'].values])
LAT_M, LON_M = 111_000, 73_000

# --- streetuse ---
df_st = pd.read_csv(os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/public_streets.csv'), quotechar='"', on_bad_lines='skip')
st_geo = df_st['geo_point_2d'].dropna().apply(ast.literal_eval)
st_coords = np.array([[d['lat'], d['lon']] for d in st_geo])
_, idx = cKDTree(st_coords).query(pave_coords)
df['streetuse'] = df_st.loc[st_geo.index, 'streetuse'].values[idx]
print('streetuse:', df['streetuse'].value_counts().to_dict())

# Encode streetuse as numeric for interaction features later
STREETUSE_WEIGHT = {'Residential': 0.3, 'Collector': 0.6, 'Arterial': 1.0}
df['streetuse_weight'] = df['streetuse'].map(STREETUSE_WEIGHT).fillna(0.5)

# --- ROW width ---
df_row = safe_load('/content/drive/MyDrive/AI-FinalProject/data/right_of_way_widths.csv')
geo_col = [c for c in df_row.columns if 'geo_point' in c.lower()][0]
width_col = [c for c in df_row.columns if 'width' in c.lower()][0]
row_geo = df_row[geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
row_coords = np.array([[d['lat'], d['lon']] for d in row_geo if isinstance(d, dict)])
row_widths = pd.to_numeric(df_row.loc[row_geo.index[:len(row_coords)], width_col], errors='coerce').fillna(0).values
_, idx = cKDTree(row_coords).query(pave_coords)
df['ROW_width'] = row_widths[idx]

# --- repair_count ---
df['repair_count'] = 0
repair_path = os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv')
if os.path.exists(repair_path):
    df_repair = safe_load('/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv')
    geo_cols = [c for c in df_repair.columns if 'geo_point' in c.lower()]
    if geo_cols:
        try:
            rg = df_repair[geo_cols[0]].dropna()
            parsed = rg.apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
            rc = np.array([[d['lat'], d['lon']] for d in parsed if isinstance(d, dict)])
            counts = cKDTree(rc * [LAT_M, LON_M]).query_ball_point(pave_coords * [LAT_M, LON_M], r=100)
            df['repair_count'] = [len(c) for c in counts]
        except: pass

# --- segment_density ---
tree_all = cKDTree(pave_coords)
neighbours = tree_all.query_ball_point(pave_coords, r=0.003)
df['segment_density'] = [len(n) - 1 for n in neighbours]

# --- elevation & slope ---
if 'elevation_m' not in df.columns:
    df['elevation_m'] = 0.0; df['slope_pct'] = 0.0
df['elevation_m'] = df['elevation_m'].fillna(df['elevation_m'].median())
df['slope_pct'] = df['slope_pct'].fillna(df['slope_pct'].median())

# --- neighbourhood ---
df['neighbourhood'] = df.get('neighbourhood', pd.Series('Unknown', index=df.index)).fillna('Unknown')

# --- traffic proximity ---
df_traffic = safe_load('/content/drive/MyDrive/AI-FinalProject/data/directional_traffic_count_locations.csv')
traf_geo_col = [c for c in df_traffic.columns if 'geo_point' in c.lower()][0]
traf_geo = df_traffic[traf_geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
traf_coords = np.array([[d['lat'], d['lon']] for d in traf_geo if isinstance(d, dict)])
traf_tree = cKDTree(traf_coords * [LAT_M, LON_M])
traf_dists, _ = traf_tree.query(pave_coords * [LAT_M, LON_M])
df['dist_to_traffic_count'] = traf_dists
df['near_traffic_counter'] = (df['dist_to_traffic_count'] <= 200).astype(int)

# --- construction proximity ---
if os.path.exists(repair_path):
    proj_geo = df_repair[geo_cols[0]].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
    proj_coords = np.array([[d['lat'], d['lon']] for d in proj_geo if isinstance(d, dict)])
    if len(proj_coords) > 0:
        proj_near = cKDTree(proj_coords * [LAT_M, LON_M]).query_ball_point(pave_coords * [LAT_M, LON_M], r=300)
        df['near_construction'] = [len(p) for p in proj_near]
    else:
        df['near_construction'] = 0
else:
    df['near_construction'] = 0

print(f'Static features done. Shape: {df.shape}')

streetuse: {'Arterial': 5877, 'Residential': 5180, 'Collector': 2707}
Static features done. Shape: (13764, 22)


## 3. Train/Val/Test Split

In [ ]:
y = df['risk_label'].values
indices = np.arange(len(df))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])
print(f'Train: {len(idx_train):,}  Val: {len(idx_val):,}  Test: {len(idx_test):,}')
print(f'Train labels: {np.bincount(y[idx_train])}')
print(f'Val   labels: {np.bincount(y[idx_val])}')
print(f'Test  labels: {np.bincount(y[idx_test])}')

Train: 9,634  Val: 2,065  Test: 2,065
Train labels: [4256 2142 3236]
Val   labels: [912 459 694]
Test  labels: [912 460 693]


## 4. Temporal Features — Weather + Filtered 311
**Key: pavement-related 311 only, survey years only.**

In [ ]:
# --- Weather ---
df_w = safe_load('/content/drive/MyDrive/AI-FinalProject/data/weather_vancouver.csv')
date_col = [c for c in df_w.columns if 'date' in c.lower() or 'time' in c.lower()][0]
df_w['date'] = pd.to_datetime(df_w[date_col], errors='coerce')
df_w = df_w.dropna(subset=['date'])
col_map = {}
for c in df_w.columns:
    cl = c.lower().strip()
    if 'max temp' in cl and 'flag' not in cl: col_map[c] = 'max_temp'
    elif 'min temp' in cl and 'flag' not in cl: col_map[c] = 'min_temp'
    elif 'total precip' in cl and 'flag' not in cl: col_map[c] = 'total_precip'
    elif 'total snow' in cl and 'flag' not in cl: col_map[c] = 'total_snow'
df_w = df_w.rename(columns=col_map)
for c in ['max_temp','min_temp','total_precip','total_snow']:
    if c in df_w.columns: df_w[c] = pd.to_numeric(df_w[c], errors='coerce').fillna(0)
df_w['freeze_thaw'] = ((df_w['max_temp'] > 0) & (df_w['min_temp'] < 0)).astype(int)
df_w['extreme'] = (df_w['total_precip'] > df_w['total_precip'].quantile(0.95)).astype(int)
df_w['year'], df_w['month'] = df_w['date'].dt.year, df_w['date'].dt.month
monthly_wx = df_w.groupby(['year','month']).agg(
    avg_max_temp=('max_temp','mean'), avg_min_temp=('min_temp','mean'),
    total_precip_mm=('total_precip','sum'), total_snow_cm=('total_snow','sum'),
    freeze_thaw_days=('freeze_thaw','sum'), extreme_days=('extreme','sum'),
).reset_index()
print(f'Weather: {len(df_w):,} days')

# --- 311: pavement-related only ---
print('\nLoading 311 (pavement-related only)...')
df_311 = safe_load('311_service_requests_2009_2021.csv', low_memory=False)
id_cols = [c for c in df_311.columns if 'case' in c.lower() or 'id' in c.lower()]
if id_cols: df_311 = df_311.drop_duplicates(subset=id_cols[0])

PAVEMENT_TYPES = [
    'Pothole Case', 'Street Repair Case', 'Sidewalk Repair Case',
    'Pavement Marking Maintenance Case', 'Pavement Markings Case',
    'Street Surface Water Flooding Case', 'Street Construction Concern Case',
    'Street Cleaning and Debris Pick Up Case', 'Curb Ramp Request Case',
]
type_col = 'Service request type'
df_311 = df_311[df_311[type_col].isin(PAVEMENT_TYPES)]
print(f'  After pavement filter: {len(df_311):,}')

date_cols = [c for c in df_311.columns if 'open' in c.lower() or 'timestamp' in c.lower() or 'created' in c.lower()]
if not date_cols: date_cols = [c for c in df_311.columns if 'date' in c.lower()]
df_311['date'] = pd.to_datetime(df_311[date_cols[0]], errors='coerce', utc=True)

lat_col = [c for c in df_311.columns if c.lower() in ['latitude','lat']]
lon_col = [c for c in df_311.columns if c.lower() in ['longitude','lon']]
if lat_col and lon_col:
    df_311['c_lat'] = pd.to_numeric(df_311[lat_col[0]], errors='coerce')
    df_311['c_lon'] = pd.to_numeric(df_311[lon_col[0]], errors='coerce')
else:
    geo_col = [c for c in df_311.columns if 'geo_local' in c.lower() or 'geo_point' in c.lower()]
    if geo_col:
        raw = df_311[geo_col[0]].dropna()
        if len(raw) > 0 and '{' in str(raw.iloc[0]):
            parsed = raw.apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
            df_311.loc[raw.index, 'c_lat'] = parsed.apply(lambda d: d.get('lat') if isinstance(d, dict) else None)
            df_311.loc[raw.index, 'c_lon'] = parsed.apply(lambda d: d.get('lon') if isinstance(d, dict) else None)

df_311 = df_311.dropna(subset=['date','c_lat','c_lon'])
df_311['year'], df_311['month'] = df_311['date'].dt.year, df_311['date'].dt.month
survey_years = set(df['Year'].unique())
df_311 = df_311[df_311['year'].isin(survey_years)]
print(f'  Filtered to survey years {survey_years}: {len(df_311):,}')

pave_m = pave_coords * [LAT_M, LON_M]
pave_tree = cKDTree(pave_m)
c311_m = np.column_stack([df_311['c_lat'].values * LAT_M, df_311['c_lon'].values * LON_M])
dists, seg_idx = pave_tree.query(c311_m)
df_311['seg_idx'] = seg_idx
df_311 = df_311[dists <= SNAP_RADIUS_M]
print(f'  Within {SNAP_RADIUS_M}m: {len(df_311):,}')

df_311['is_pothole'] = (df_311[type_col] == 'Pothole Case').astype(int)
complaint_monthly = df_311.groupby(['seg_idx','year','month']).agg(
    cnt=('seg_idx','size'), pothole_cnt=('is_pothole','sum')
).reset_index()
print(f'  Monthly groups: {len(complaint_monthly):,}')

Weather: 2,557 days

Loading 311 (pavement-related only)...
  After pavement filter: 135,647
  Filtered to survey years {np.int64(2020), np.int64(2021)}: 27,187
  Within 150m: 27,024
  Monthly groups: 22,807


## 5. Build Temporal Tensor — ★ WITH INTERACTION CHANNELS ★
**v3 Key Change:** Weather is city-wide → same for all segments → BiLSTM can't distinguish.

**Fix:** Multiply weather by segment-specific attributes:
- `precip × slope` → steep roads get more water damage
- `freeze_thaw × streetuse_weight` → arterials get more freeze-thaw traffic stress
- `snow × slope` → snow accumulation worse on slopes

This gives **each segment a unique temporal signature**, even though raw weather is global.

| Channel | Feature | Segment-specific? |
|---------|---------|-------------------|
| 0 | avg_max_temp | ❌ global |
| 1 | avg_min_temp | ❌ global |
| 2 | total_precip_mm | ❌ global |
| 3 | freeze_thaw_days | ❌ global |
| 4 | complaint_count | ✅ |
| 5 | complaint_density | ✅ |
| 6 | pothole_count | ✅ |
| 7 | **precip × slope** | ✅ NEW |
| 8 | **freeze_thaw × streetuse_weight** | ✅ NEW |
| 9 | **snow × slope** | ✅ NEW |
| 10 | **temp_range × streetuse_weight** | ✅ NEW |
| 11 | extreme_days | ❌ global |

In [ ]:
# --- Build temporal tensor (N, 12, 12) — 4 interaction channels ---
N = len(df)
N_FEAT = 12
X_temporal = np.zeros((N, 12, N_FEAT), dtype=np.float32)
years = df['Year'].values
slopes = df['slope_pct'].values
su_weights = df['streetuse_weight'].values

# Neighbourhood-month totals for complaint density
seg_to_neigh = df['neighbourhood'].values
neigh_month_tot = {}
for _, r in complaint_monthly.iterrows():
    key = (seg_to_neigh[int(r['seg_idx'])], int(r['year']), int(r['month']))
    neigh_month_tot[key] = neigh_month_tot.get(key, 0) + r['cnt']

for i in range(N):
    yr = years[i]
    sl = slopes[i]
    sw = su_weights[i]
    wx = monthly_wx[monthly_wx['year'] == yr]

    for _, r in wx.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            # --- Original global weather (4 channels) ---
            X_temporal[i,m,0] = r['avg_max_temp']
            X_temporal[i,m,1] = r['avg_min_temp']
            X_temporal[i,m,2] = r['total_precip_mm']
            X_temporal[i,m,3] = r['freeze_thaw_days']
            X_temporal[i,m,11] = r['extreme_days']

            # --- ★ INTERACTION CHANNELS (segment-specific!) ★ ---
            X_temporal[i,m,7]  = r['total_precip_mm'] * sl          # precip × slope
            X_temporal[i,m,8]  = r['freeze_thaw_days'] * sw         # freeze_thaw × road importance
            X_temporal[i,m,9]  = r['total_snow_cm'] * sl             # snow × slope
            X_temporal[i,m,10] = (r['avg_max_temp'] - r['avg_min_temp']) * sw  # temp_range × road importance

    # --- Complaint channels (already segment-specific) ---
    seg_c = complaint_monthly[(complaint_monthly['seg_idx']==i) & (complaint_monthly['year']==yr)]
    neigh = seg_to_neigh[i]
    for _, r in seg_c.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            X_temporal[i,m,4] = r['cnt']           # complaint count
            nt = neigh_month_tot.get((neigh, yr, int(r['month'])), 1)
            X_temporal[i,m,5] = min(r['cnt'] / max(nt, 1), 1.0)  # density
            X_temporal[i,m,6] = r['pothole_cnt']    # pothole count

# Static aggregates
df['complaint_total'] = X_temporal[:,:,4].sum(axis=1)
df['pothole_total'] = X_temporal[:,:,6].sum(axis=1)

print(f'X_temporal: {X_temporal.shape}')
print(f'Nonzero complaint-months: {(X_temporal[:,:,4]>0).sum():,}')
print(f'Nonzero interaction-months (precip×slope): {(X_temporal[:,:,7]>0).sum():,}')
print(f'complaint_total: mean={df["complaint_total"].mean():.1f}, max={df["complaint_total"].max():.0f}')

X_temporal: (13764, 12, 12)
Nonzero complaint-months: 11,364
Nonzero interaction-months (precip×slope): 159,376
complaint_total: mean=1.0, max=32


## 6. Assemble Branch Tensors & Normalize
> `neigh_high_risk_pct` and `neigh_mean_pci` **removed** (target leakage).

In [ ]:
# --- Road-MLP: 11 dims ---
su_dummies = pd.get_dummies(df['streetuse'], prefix='su')
for c in ['su_Arterial','su_Collector','su_Residential']:
    if c not in su_dummies.columns: su_dummies[c] = 0

X_road_raw = np.column_stack([
    df['length_(m)'].fillna(df['length_(m)'].median()).values,
    su_dummies[['su_Arterial','su_Collector','su_Residential']].values,
    df['elevation_m'].values,
    df['slope_pct'].values,
    df['repair_count'].values,
    df['segment_density'].values,
    df['source'].values,
    df['dist_to_traffic_count'].values,
    df['near_construction'].values,
]).astype(np.float32)
print(f'Road-MLP: {X_road_raw.shape[1]} dims')

# --- Tabular-MLP: ~27 dims (NO leaky features) ---
neigh_dummies = pd.get_dummies(df['neighbourhood'], prefix='n')
X_tab_raw = np.column_stack([
    neigh_dummies.values,
    df['ROW_width'].values,
    df['complaint_total'].values,
    df['pothole_total'].values,
    df['near_traffic_counter'].values,
]).astype(np.float32)
print(f'Tabular-MLP: {X_tab_raw.shape[1]} dims')

# --- Normalize (fit on train only) ---
sc_road = StandardScaler().fit(X_road_raw[idx_train])
sc_tab = StandardScaler().fit(X_tab_raw[idx_train])
sc_temp = StandardScaler().fit(X_temporal[idx_train].reshape(-1, N_FEAT))

def split_norm(X, sc, idxs, seq=False, nf=None):
    if seq:
        n = len(idxs)
        return sc.transform(X[idxs].reshape(-1, nf)).reshape(n, 12, nf).astype(np.float32)
    return sc.transform(X[idxs]).astype(np.float32)

Xr_tr = split_norm(X_road_raw, sc_road, idx_train)
Xr_v  = split_norm(X_road_raw, sc_road, idx_val)
Xr_te = split_norm(X_road_raw, sc_road, idx_test)
Xt_tr = split_norm(X_temporal, sc_temp, idx_train, True, N_FEAT)
Xt_v  = split_norm(X_temporal, sc_temp, idx_val, True, N_FEAT)
Xt_te = split_norm(X_temporal, sc_temp, idx_test, True, N_FEAT)
Xb_tr = split_norm(X_tab_raw, sc_tab, idx_train)
Xb_v  = split_norm(X_tab_raw, sc_tab, idx_val)
Xb_te = split_norm(X_tab_raw, sc_tab, idx_test)
y_tr, y_v, y_te = y[idx_train], y[idx_val], y[idx_test]

print(f'Road: {Xr_tr.shape}, Temporal: {Xt_tr.shape}, Tabular: {Xb_tr.shape}')

Road-MLP: 11 dims
Tabular-MLP: 27 dims
Road: (9634, 11), Temporal: (9634, 12, 12), Tabular: (9634, 27)


## 7. ★ Model Definitions (v3 Changes) ★

### What changed:
1. **`TemporalCNN`** replaces `BiLSTMBranch` — 1D convolutions on 12-step sequence, output 32d (not 128d)
2. **`GatedFusion`** replaces naive concat — learns per-sample branch importance weights
3. **No auxiliary regression head** — PCI score prediction removed (was predicting the label in disguise)

In [ ]:
# ============================================================
# Loss Functions
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha, self.gamma, self.label_smoothing = alpha, gamma, label_smoothing
    def forward(self, logits, targets):
        n_classes = logits.size(1)
        if self.label_smoothing > 0:
            with torch.no_grad():
                smooth = torch.full_like(logits, self.label_smoothing / (n_classes - 1))
                smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
            log_p = torch.log_softmax(logits, dim=1)
            loss = -(smooth * log_p).sum(dim=1)
        else:
            loss = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.softmax(logits, dim=1).gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = (1 - p_t) ** self.gamma * loss
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean()


class OrdinalPenalty(nn.Module):
    def __init__(self, weight=0.3):
        super().__init__()
        self.weight = weight
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        classes = torch.arange(logits.size(1), device=logits.device, dtype=torch.float32)
        expected = (probs * classes.unsqueeze(0)).sum(dim=1)
        return self.weight * ((expected - targets.float()) ** 2).mean()


# ============================================================
# Branch 1: Road-MLP (same as v2)
# ============================================================
class RoadMLP(nn.Module):
    def __init__(self, dim, emb=16, hidden=128, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.enc = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, 3)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))


# ============================================================
# Branch 2: ★ Temporal 1D-CNN (replaces BiLSTM) ★
# ============================================================
class TemporalCNN(nn.Module):
    """1D-CNN for 12-step temporal sequence.

    Why CNN over BiLSTM?
    - 12 steps is too short for LSTM to learn meaningful hidden states
    - CNN kernels naturally capture "consecutive bad weather months" patterns
    - 75% fewer params (32d emb vs 128d), less noise into fusion

    Architecture:
      Input: (B, 12, 12) → permute → (B, 12, 12) [channels=12, length=12]
      Conv1d(12→32, k=3, pad=1) → BN → ReLU → Dropout
      Conv1d(32→64, k=3, pad=1) → BN → ReLU → Dropout
      AdaptiveAvgPool1d(1) → flatten → FC(64→32) → 32d embedding
    """
    def __init__(self, in_channels=12, emb=32, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(drop),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(drop),
            nn.AdaptiveAvgPool1d(1))  # → (B, 64, 1)
        self.fc = nn.Sequential(nn.Linear(64, emb), nn.ReLU(), nn.Dropout(drop))
        self.head = nn.Linear(emb, 3)

    def get_embedding(self, x):
        # x: (B, 12, 12) = (batch, time_steps, features)
        x = x.permute(0, 2, 1)  # → (B, features, time_steps) for Conv1d
        x = self.conv(x).squeeze(-1)  # → (B, 64)
        return self.fc(x)  # → (B, 32)

    def forward(self, x):
        return self.head(self.get_embedding(x))


# ============================================================
# Branch 3: Tabular-MLP (same as v2)
# ============================================================
class TabularMLP(nn.Module):
    def __init__(self, dim, emb=16, hidden=128, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.enc = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, 3)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))


# ============================================================
# ★ Gated Attention Fusion ★
# ============================================================
class GatedFusion(nn.Module):
    """Attention-weighted late fusion.

    Instead of naive concat (which lets 32d CNN noise mix equally with 16d MLP signal),
    this learns per-sample importance weights for each branch.

    Flow:
      1. Project each branch embedding to uniform 64d
      2. Concat all → gate network → softmax → 3 weights (one per branch)
      3. Weighted sum of projected embeddings
      4. Classification head on fused representation

    Key insight: if temporal branch is useless for a given sample,
    the gate will assign it near-zero weight automatically.
    """
    def __init__(self, road, temporal, tabular, proj_dim=64, drop=0.4):
        super().__init__()
        self.road, self.temporal, self.tabular = road, temporal, tabular

        # Project each branch to uniform dimension
        r_emb = road.emb_dim      # 16
        t_emb = temporal.emb_dim   # 32
        b_emb = tabular.emb_dim    # 16

        self.proj_road = nn.Linear(r_emb, proj_dim)
        self.proj_temp = nn.Linear(t_emb, proj_dim)
        self.proj_tab  = nn.Linear(b_emb, proj_dim)

        # Gate: learns which branch matters for each sample
        total_emb = r_emb + t_emb + b_emb  # 16+32+16 = 64
        self.gate = nn.Sequential(
            nn.Linear(total_emb, 32), nn.ReLU(),
            nn.Linear(32, 3),  # 3 branch weights
            nn.Softmax(dim=1))

        # Classification head on fused representation
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(64, 3))

    def forward(self, xr, xt, xb):
        # Get branch embeddings
        er = self.road.get_embedding(xr)      # (B, 16)
        et = self.temporal.get_embedding(xt)   # (B, 32)
        eb = self.tabular.get_embedding(xb)    # (B, 16)

        # Gate weights
        concat = torch.cat([er, et, eb], dim=1)  # (B, 64)
        weights = self.gate(concat)  # (B, 3)

        # Project to uniform dim
        pr = self.proj_road(er)   # (B, 64)
        pt = self.proj_temp(et)   # (B, 64)
        pb = self.proj_tab(eb)    # (B, 64)

        # Weighted sum
        fused = (weights[:, 0:1] * pr +
                 weights[:, 1:2] * pt +
                 weights[:, 2:3] * pb)  # (B, 64)

        return self.classifier(fused)

    def forward_ablation(self, xr, xt, xb, disable=None):
        er = self.road.get_embedding(xr) if disable != 'road' else torch.zeros(xr.size(0), self.road.emb_dim, device=xr.device)
        et = self.temporal.get_embedding(xt) if disable != 'temporal' else torch.zeros(xt.size(0), self.temporal.emb_dim, device=xt.device)
        eb = self.tabular.get_embedding(xb) if disable != 'tabular' else torch.zeros(xb.size(0), self.tabular.emb_dim, device=xb.device)
        concat = torch.cat([er, et, eb], dim=1)
        weights = self.gate(concat)
        pr, pt, pb = self.proj_road(er), self.proj_temp(et), self.proj_tab(eb)
        fused = weights[:,0:1]*pr + weights[:,1:2]*pt + weights[:,2:3]*pb
        return self.classifier(fused)

    def get_gate_weights(self, xr, xt, xb):
        """Return gate weights for analysis."""
        with torch.no_grad():
            er = self.road.get_embedding(xr)
            et = self.temporal.get_embedding(xt)
            eb = self.tabular.get_embedding(xb)
            return self.gate(torch.cat([er, et, eb], dim=1))


# ============================================================
# Dataset
# ============================================================
class FusionDS(Dataset):
    def __init__(self, xr, xt, xb, y_cls):
        self.xr, self.xt, self.xb, self.y_cls = xr, xt, xb, y_cls
    def __len__(self): return len(self.y_cls)
    def __getitem__(self, i):
        return self.xr[i], self.xt[i], self.xb[i], self.y_cls[i]

# Class weights
cc = np.bincount(y_tr)
CW = torch.FloatTensor((1.0/cc) / (1.0/cc).sum() * 3).to(DEVICE)
print(f'Class weights: {CW.cpu().numpy()}')

# Count params
r = RoadMLP(dim=Xr_tr.shape[1])
t = TemporalCNN(in_channels=N_FEAT)
b = TabularMLP(dim=Xb_tr.shape[1])
print(f'\nParam count:')
print(f'  Road-MLP:     {sum(p.numel() for p in r.parameters()):,}')
print(f'  TemporalCNN:  {sum(p.numel() for p in t.parameters()):,}')
print(f'  Tabular-MLP:  {sum(p.numel() for p in b.parameters()):,}')
g = GatedFusion(r, t, b)
print(f'  GatedFusion:  {sum(p.numel() for p in g.parameters()):,} (total)')
del r, t, b, g

Class weights: [0.6973287 1.3855419 0.9171294]

Param count:
  Road-MLP:     11,139
  TemporalCNN:  9,763
  Tabular-MLP:  13,187
  GatedFusion:  45,039 (total)


## 8. Training Utilities

In [ ]:
def train_branch(model, X_tr, X_v, y_tr_, y_v_, epochs=60, bs=256, lr=2e-3,
                 patience=12, criterion=None, verbose=True):
    model = model.to(DEVICE)
    tr_dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_).long()), batch_size=bs, shuffle=True)
    v_dl  = DataLoader(TensorDataset(torch.tensor(X_v), torch.tensor(y_v_).long()), batch_size=bs)
    if criterion is None:
        criterion = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
    ordinal = OrdinalPenalty(weight=0.3)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, wait, best_sd = 0, 0, None
    for ep in range(1, epochs+1):
        model.train()
        for X, yy in tr_dl:
            X, yy = X.to(DEVICE), yy.to(DEVICE)
            logits = model(X)
            loss = criterion(logits, yy) + ordinal(logits, yy)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for X, yy in v_dl:
                preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
                labs.extend(yy.numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: wait += 1
        if verbose and ep % 10 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}')
        if wait >= patience:
            if verbose: print(f'  Early stop ep {ep}')
            break
    model.load_state_dict(best_sd)
    return model, best_f1


def eval_test(model, X_te, y_te_, bs=256):
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te_).long()), batch_size=bs)
    preds, labs = [], []
    with torch.no_grad():
        for X, yy in dl:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

print('Training functions ready.')

Training functions ready.


## 9. Pre-train Individual Branches

In [ ]:
print('=== Road-MLP (16d embedding) ===')
road_model = RoadMLP(dim=Xr_tr.shape[1])
road_model, road_val_f1 = train_branch(road_model, Xr_tr, Xr_v, y_tr, y_v)
road_test_f1, _, _ = eval_test(road_model, Xr_te, y_te)
print(f'  Road-MLP: val={road_val_f1:.4f} test={road_test_f1:.4f}\n')

print('=== TemporalCNN (32d embedding) ===')
temp_model = TemporalCNN(in_channels=N_FEAT, emb=32)
temp_model, temp_val_f1 = train_branch(temp_model, Xt_tr, Xt_v, y_tr, y_v)
temp_test_f1, _, _ = eval_test(temp_model, Xt_te, y_te)
print(f'  TemporalCNN: val={temp_val_f1:.4f} test={temp_test_f1:.4f}\n')

print('=== Tabular-MLP (16d embedding) ===')
tab_model = TabularMLP(dim=Xb_tr.shape[1])
tab_model, tab_val_f1 = train_branch(tab_model, Xb_tr, Xb_v, y_tr, y_v)
tab_test_f1, _, _ = eval_test(tab_model, Xb_te, y_te)
print(f'  Tabular-MLP: val={tab_val_f1:.4f} test={tab_test_f1:.4f}\n')

print('--- Unimodal Summary ---')
print(f'{"Branch":<15} {"Val F1":>8} {"Test F1":>8}')
print('-'*33)
for name, vf, tf in [('Road-MLP',road_val_f1,road_test_f1),
                      ('TemporalCNN',temp_val_f1,temp_test_f1),
                      ('Tabular-MLP',tab_val_f1,tab_test_f1)]:
    print(f'{name:<15} {vf:>8.4f} {tf:>8.4f}')

=== Road-MLP (16d embedding) ===
  Ep  10 val_f1=0.4100 best=0.4100
  Ep  20 val_f1=0.4138 best=0.4212
  Ep  30 val_f1=0.4175 best=0.4264
  Ep  40 val_f1=0.4185 best=0.4304
  Early stop ep 48
  Road-MLP: val=0.4304 test=0.4185

=== TemporalCNN (32d embedding) ===
  Ep  10 val_f1=0.3725 best=0.3843
  Ep  20 val_f1=0.3715 best=0.3906
  Early stop ep 26
  TemporalCNN: val=0.3906 test=0.3632

=== Tabular-MLP (16d embedding) ===
  Ep  10 val_f1=0.3891 best=0.3953
  Ep  20 val_f1=0.3927 best=0.4110
  Early stop ep 25
  Tabular-MLP: val=0.4110 test=0.3894

--- Unimodal Summary ---
Branch            Val F1  Test F1
---------------------------------
Road-MLP          0.4304   0.4185
TemporalCNN       0.3906   0.3632
Tabular-MLP       0.4110   0.3894


## 10. ★ Gated Fusion Training ★

In [ ]:
fusion = GatedFusion(road_model, temp_model, tab_model).to(DEVICE)
print(f'GatedFusion total params: {sum(p.numel() for p in fusion.parameters()):,}')

tr_dl = DataLoader(FusionDS(torch.tensor(Xr_tr), torch.tensor(Xt_tr),
                             torch.tensor(Xb_tr), torch.tensor(y_tr).long()),
                   batch_size=256, shuffle=True)
v_dl = DataLoader(FusionDS(torch.tensor(Xr_v), torch.tensor(Xt_v),
                            torch.tensor(Xb_v), torch.tensor(y_v).long()), batch_size=256)
te_dl = DataLoader(FusionDS(torch.tensor(Xr_te), torch.tensor(Xt_te),
                             torch.tensor(Xb_te), torch.tensor(y_te).long()), batch_size=256)

focal = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
ordinal = OrdinalPenalty(weight=0.3)

def eval_fusion(model, dl, disable=None):
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for xr,xt,xb,yy in dl:
            xr,xt,xb = xr.to(DEVICE), xt.to(DEVICE), xb.to(DEVICE)
            logits = model.forward_ablation(xr,xt,xb,disable) if disable else model(xr,xt,xb)
            preds.extend(logits.argmax(1).cpu().numpy())
            labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

# Phase 1: Warmup (branches frozen, only train gate + projections + classifier)
print('\n--- Phase 1: Warmup (5 ep, branches frozen) ---')
for p in list(fusion.road.parameters()) + list(fusion.temporal.parameters()) + list(fusion.tabular.parameters()):
    p.requires_grad = False
opt = torch.optim.AdamW([p for p in fusion.parameters() if p.requires_grad], lr=2e-3, weight_decay=1e-4)
for ep in range(1, 6):
    fusion.train()
    for xr,xt,xb,yy in tr_dl:
        xr,xt,xb,yy = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE)
        logits = fusion(xr,xt,xb)
        loss = focal(logits, yy) + ordinal(logits, yy)
        opt.zero_grad(); loss.backward(); opt.step()
    f1, _, _ = eval_fusion(fusion, v_dl)
    print(f'  Warmup {ep}/5 val_f1={f1:.4f}')

# Phase 2: End-to-end (all params unfrozen, lower lr)
print('\n--- Phase 2: End-to-end (60 ep) ---')
for p in fusion.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(fusion.parameters(), lr=2e-4, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
best_f1, wait, best_sd = 0, 0, None
for ep in range(1, 61):
    fusion.train()
    for xr,xt,xb,yy in tr_dl:
        xr,xt,xb,yy = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE)
        logits = fusion(xr,xt,xb)
        loss = focal(logits, yy) + ordinal(logits, yy)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(fusion.parameters(), 1.0); opt.step()
    sch.step()
    f1, _, _ = eval_fusion(fusion, v_dl)
    mk = ''
    if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in fusion.state_dict().items()}; mk = ' *'
    else: wait += 1
    if ep % 5 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}{mk}')
    if wait >= 12: print(f'  Early stop ep {ep}'); break

fusion.load_state_dict(best_sd)
fusion = fusion.to(DEVICE)

GatedFusion total params: 45,039

--- Phase 1: Warmup (5 ep, branches frozen) ---
  Warmup 1/5 val_f1=0.4266
  Warmup 2/5 val_f1=0.4321
  Warmup 3/5 val_f1=0.4359
  Warmup 4/5 val_f1=0.4340
  Warmup 5/5 val_f1=0.4356

--- Phase 2: End-to-end (60 ep) ---
  Ep   5 val_f1=0.4329 best=0.4348
  Ep  10 val_f1=0.4329 best=0.4348
  Ep  15 val_f1=0.4318 best=0.4350
  Ep  20 val_f1=0.4347 best=0.4350
  Ep  25 val_f1=0.4270 best=0.4350
  Early stop ep 26


## 11. Test Evaluation, Ablation & Gate Analysis

In [ ]:
test_f1, preds, labs = eval_fusion(fusion, te_dl)
print(f'\n{"="*60}')
print(f'FUSION v3 TEST MACRO F1: {test_f1:.4f}  (val best: {best_f1:.4f})')
print(f'{"="*60}')
print(classification_report(labs, preds, target_names=['Low','Medium','High']))

# Ablation
print(f'{"="*60}')
print('ABLATION STUDY')
print(f'{"="*60}')
results = {'Full model': test_f1}
for branch in ['road','temporal','tabular']:
    ab_f1, _, _ = eval_fusion(fusion, te_dl, disable=branch)
    results[f'w/o {branch}'] = ab_f1

print(f'\n{"Config":<20} {"F1":>6} {"Delta":>8}')
print('-'*36)
for k, v in results.items():
    d = f'{v - test_f1:+.4f}' if k != 'Full model' else '  base'
    print(f'{k:<20} {v:>6.4f} {d:>8}')

# --- Gate Weight Analysis ---
print(f'\n{"="*60}')
print('GATE WEIGHT ANALYSIS')
print(f'{"="*60}')
all_weights = []
with torch.no_grad():
    for xr,xt,xb,yy in te_dl:
        w = fusion.get_gate_weights(xr.to(DEVICE), xt.to(DEVICE), xb.to(DEVICE))
        all_weights.append(w.cpu().numpy())
all_weights = np.vstack(all_weights)

print(f'\nMean gate weights across test set:')
print(f'  Road-MLP:     {all_weights[:,0].mean():.4f} ± {all_weights[:,0].std():.4f}')
print(f'  TemporalCNN:  {all_weights[:,1].mean():.4f} ± {all_weights[:,1].std():.4f}')
print(f'  Tabular-MLP:  {all_weights[:,2].mean():.4f} ± {all_weights[:,2].std():.4f}')

# Gate weights per true class
print(f'\nGate weights by true class:')
print(f'{"Class":<10} {"Road":>8} {"Temporal":>10} {"Tabular":>10}')
print('-'*40)
for cls, name in enumerate(['Low','Medium','High']):
    mask = labs == cls
    if mask.sum() > 0:
        print(f'{name:<10} {all_weights[mask,0].mean():>8.4f} {all_weights[mask,1].mean():>10.4f} {all_weights[mask,2].mean():>10.4f}')

print(f'\n--- Fusion v3 vs Unimodal ---')
best_uni = max(road_test_f1, temp_test_f1, tab_test_f1)
print(f'Best unimodal: {best_uni:.4f}')
print(f'Fusion v3:     {test_f1:.4f}')
print(f'Improvement:   {test_f1 - best_uni:+.4f}')


FUSION v3 TEST MACRO F1: 0.4335  (val best: 0.4350)
              precision    recall  f1-score   support

         Low       0.56      0.54      0.55       912
      Medium       0.27      0.29      0.28       460
        High       0.47      0.47      0.47       693

    accuracy                           0.46      2065
   macro avg       0.43      0.43      0.43      2065
weighted avg       0.47      0.46      0.46      2065

ABLATION STUDY

Config                   F1    Delta
------------------------------------
Full model           0.4335     base
w/o road             0.3545  -0.0790
w/o temporal         0.4325  -0.0010
w/o tabular          0.4178  -0.0157

GATE WEIGHT ANALYSIS

Mean gate weights across test set:
  Road-MLP:     0.4121 ± 0.0460
  TemporalCNN:  0.2951 ± 0.0419
  Tabular-MLP:  0.2928 ± 0.0287

Gate weights by true class:
Class          Road   Temporal    Tabular
----------------------------------------
Low          0.4177     0.2904     0.2920
Medium       0.4073 

## 12. XGBoost Baseline + Ensemble + Final Comparison

In [ ]:
try:
    from xgboost import XGBClassifier
except ImportError:
    !pip install -q xgboost
    from xgboost import XGBClassifier

X_flat_tr = np.concatenate([Xr_tr, Xt_tr.reshape(len(Xr_tr),-1), Xb_tr], axis=1)
X_flat_v  = np.concatenate([Xr_v,  Xt_v.reshape(len(Xr_v),-1),  Xb_v],  axis=1)
X_flat_te = np.concatenate([Xr_te, Xt_te.reshape(len(Xr_te),-1), Xb_te], axis=1)

xgb = XGBClassifier(n_estimators=500, max_depth=7, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                     eval_metric='mlogloss', random_state=42, verbosity=0)
xgb.fit(X_flat_tr, y_tr, eval_set=[(X_flat_v, y_v)], verbose=False)
xgb_preds = xgb.predict(X_flat_te)
xgb_f1 = f1_score(y_te, xgb_preds, average='macro')
print(f'XGBoost Test Macro F1: {xgb_f1:.4f}')
print(classification_report(y_te, xgb_preds, target_names=['Low','Medium','High']))

# Ensemble: Fusion + XGBoost
xgb_proba = xgb.predict_proba(X_flat_te)
fusion.eval()
fusion_proba = []
with torch.no_grad():
    for xr,xt,xb,yy in te_dl:
        logits = fusion(xr.to(DEVICE), xt.to(DEVICE), xb.to(DEVICE))
        fusion_proba.append(torch.softmax(logits, dim=1).cpu().numpy())
fusion_proba = np.vstack(fusion_proba)

print('\n--- Ensemble Search ---')
best_w, best_ens_f1 = 0, 0
for w in np.arange(0.1, 0.9, 0.05):
    ens = (w * fusion_proba + (1-w) * xgb_proba).argmax(1)
    ef = f1_score(y_te, ens, average='macro')
    if ef > best_ens_f1: best_w, best_ens_f1 = w, ef

ens_preds = (best_w * fusion_proba + (1-best_w) * xgb_proba).argmax(1)
print(f'Best ensemble: fusion_weight={best_w:.2f}, F1={best_ens_f1:.4f}')
print(classification_report(y_te, ens_preds, target_names=['Low','Medium','High']))

# Final comparison
print(f'\n{"="*60}')
print('FINAL COMPARISON: v1 → v2 → v3')
print(f'{"="*60}')
print(f'{"Model":<30} {"Test F1":>8}')
print('-'*40)
print(f'{"v1 Baseline":<30} {"0.3915":>8}')
print(f'{"v2 Enhanced":<30} {"0.4358":>8}')
print(f'{"---":>30}')
print(f'{"v3 Road-MLP":<30} {road_test_f1:>8.4f}')
print(f'{"v3 TemporalCNN":<30} {temp_test_f1:>8.4f}')
print(f'{"v3 Tabular-MLP":<30} {tab_test_f1:>8.4f}')
print(f'{"v3 GatedFusion":<30} {test_f1:>8.4f}')
print(f'{"v3 XGBoost":<30} {xgb_f1:>8.4f}')
print(f'{"v3 Ensemble (Fusion+XGB)":<30} {best_ens_f1:>8.4f}')
print(f'\nv1→v3 improvement: {test_f1 - 0.3915:+.4f}')
print(f'v2→v3 improvement: {test_f1 - 0.4358:+.4f}')

XGBoost Test Macro F1: 0.4495
              precision    recall  f1-score   support

         Low       0.55      0.73      0.63       912
      Medium       0.37      0.15      0.22       460
        High       0.51      0.50      0.50       693

    accuracy                           0.52      2065
   macro avg       0.48      0.46      0.45      2065
weighted avg       0.50      0.52      0.49      2065


--- Ensemble Search ---
Best ensemble: fusion_weight=0.75, F1=0.4546
              precision    recall  f1-score   support

         Low       0.57      0.66      0.61       912
      Medium       0.34      0.21      0.26       460
        High       0.48      0.51      0.50       693

    accuracy                           0.51      2065
   macro avg       0.46      0.46      0.45      2065
weighted avg       0.49      0.51      0.49      2065


FINAL COMPARISON: v1 → v2 → v3
Model                           Test F1
----------------------------------------
v1 Baseline              

---
**End of v3 pipeline.**

### What changed from v2:
1. ✅ **Interaction channels** (precip×slope, freeze_thaw×streetuse, snow×slope, temp_range×streetuse) — breaks weather uniformity
2. ✅ **1D-CNN** replaces BiLSTM — 32d embedding, fewer params, better for short sequences
3. ✅ **Gated Attention Fusion** — auto-weights branches per sample
4. ✅ **Removed PCI regression head** — was target leakage in disguise
5. ✅ **Removed `neigh_high_risk_pct`** — target leakage
6. ✅ **Gate weight analysis** — shows learned branch importance